# ANON AI ULTRA — Ministral 3 LoRA eğitimi

Bu notebook, test ettiğimiz Ministral 3 3B Instruct modelini Türkçe sohbet verisiyle ANON AI tarzına uyarlamak için QLoRA/LoRA eğitimi yapar.

**Temel model:** mistralai/Ministral-3-3B-Instruct-2512-BF16  
**Eğitim yöntemi:** 4-bit QLoRA + LoRA adaptörü  
**Veri:** depodaki Türkçe sohbet veri hazırlama betiği  
**Eğitim hedefi:** yalnızca assistant cevapları; kullanıcı ve system mesajları loss dışında tutulur.

Önemli:
- Bu notebook eski 30M modeli değiştirmez.
- Eğitilen çıktı bir LoRA adaptörüdür; tek başına tam model değildir.
- Android uygulamasına otomatik bağlanmaz.
- Ücretsiz Colab oturumu eğitim sırasında kesilebilir. Kesilirse checkpoint'lerden devam etmek için sonradan resume adımı ekleyebiliriz.
- Başlamadan önce Colab'da GPU seçili olmalı. Tesla T4 ile 4-bit ayarlar ve 1024 token bağlamı hedeflenmiştir.

Model kartı: https://huggingface.co/mistralai/Ministral-3-3B-Instruct-2512-BF16  
Transformers model desteği: https://huggingface.co/docs/transformers/model_doc/ministral3

In [ ]:
import os, sys, subprocess, torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "GPU bulunamadı")
assert torch.cuda.is_available(), "Colab > Runtime > Change runtime type > GPU seçmelisin."
print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

packages = [
    "git+https://github.com/huggingface/transformers.git",
    "peft",
    "accelerate",
    "bitsandbytes",
    "datasets",
    "safetensors",
    "mistral-common",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", *packages], check=True)
print("Kurulum tamam. Colab oturumunun yeniden başlatılması istenirse yeniden başlatıp hücreleri baştan çalıştır.")

## 1. Kütüphane sürümlerini ve GPU'yu kontrol et

İlk hücreden sonra oturum yeniden başlatıldıysa bu hücreyi çalıştır. T4 için FP16 hesaplama kullanıyoruz.

In [ ]:
import torch, transformers, peft, datasets, bitsandbytes
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("datasets:", datasets.__version__)
print("bitsandbytes:", bitsandbytes.__version__)
print("GPU:", torch.cuda.get_device_name(0))
assert torch.cuda.is_available()

## 2. Depoyu ve Türkçe sohbet verisini hazırla

Önceki veri hazırlama betiğini kullanıyoruz. İlk eğitim denemesinde 6.000 konuşmayla başlayacağız; daha sonra kalite ve GPU süresine göre artırabiliriz.

In [ ]:
import os, subprocess, sys
REPO_DIR = "/content/ANON-AI-ULTRA"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/furkanmeyra2173-create/ANON-AI-ULTRA.git", REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)

os.chdir(REPO_DIR)
DATA_PATH = "/content/anon_ai_turkish_chat.jsonl"
MAX_CONVERSATIONS = 6000

command = [
    sys.executable, "training/prepare_instruction_data.py",
    "--dataset-id", "cisimcik/turkish-chat-max-25k",
    "--config", "all",
    "--split", "train",
    "--language-mode", "tr_only",
    "--prompt-language", "tr",
    "--max-samples", str(MAX_CONVERSATIONS),
    "--output", DATA_PATH,
]
subprocess.run(command, check=True)
print("Veri satırı:", sum(1 for _ in open(DATA_PATH, encoding="utf-8")))
print("Veri dosyası:", DATA_PATH)

## 3. Ministral 3 modelini 4-bit yükle

Modelin metin ve görsel bileşenleri olan tam mimarisini yüklüyoruz. Eğitimde LoRA hedeflerini yalnızca dil modelindeki attention ve MLP katmanlarına uygulayacağız.

In [ ]:
import torch
from transformers import Mistral3ForConditionalGeneration, MistralCommonBackend, BitsAndBytesConfig

MODEL_ID = "mistralai/Ministral-3-3B-Instruct-2512-BF16"

tokenizer = MistralCommonBackend.from_pretrained(MODEL_ID)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = Mistral3ForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    torch_dtype=torch.float16,
    device_map="auto",
)
model.config.use_cache = False
if hasattr(model.config, "text_config"):
    model.config.text_config.use_cache = False

print("Yüklenen model:", MODEL_ID)
print("Tokenizer:", type(tokenizer).__name__)
print("GPU:", torch.cuda.get_device_name(0))
print("Model yükleme tamamlandı.")

## 4. ANON AI sistem karakteri ve eğitim örnekleri

Her sohbetin başına ANON AI karakter yönergesini ekliyoruz. Her assistant mesajı ayrı eğitim hedefi olur. Önceki konuşma prompt olarak kalır; yalnızca o assistant cevabının token'ları eğitilir.

In [ ]:
import json, random
from datasets import Dataset

ANON_SYSTEM = (
    "Sen ANON AI'sin. Türkçe konuş ve açık, doğal, yardımcı cevaplar ver. "
    "Kullanıcı başka bir dil istemedikçe Türkçe yanıtla. "
    "Bilmediğin bir konuda uydurma; emin olmadığını açıkça belirt. "
    "Gereksiz tekrar yapma. Sorunun gerektirdiği kadar ayrıntı ver."
)

def to_ids(value):
    if isinstance(value, dict):
        value = value["input_ids"]
    if torch.is_tensor(value):
        value = value.tolist()
    if value and isinstance(value[0], list):
        value = value[0]
    return [int(x) for x in value]

def normalize_chat(raw_messages):
    clean = []
    original_system = []
    for item in raw_messages:
        role = str(item.get("role", "")).strip().lower()
        content = item.get("content")
        if role not in ("system", "user", "assistant") or not isinstance(content, str):
            continue
        content = content.strip()
        if not content:
            continue
        if role == "system":
            original_system.append(content)
        else:
            clean.append({"role": role, "content": content})

    system_text = ANON_SYSTEM
    if original_system:
        system_text += "\n\nEk sistem yönergesi:\n" + "\n".join(original_system)
    return [{"role": "system", "content": system_text}] + clean

with open(DATA_PATH, encoding="utf-8") as f:
    conversations = [json.loads(line)["messages"] for line in f if line.strip()]

random.Random(42).shuffle(conversations)
eval_count = max(1, int(len(conversations) * 0.05))
eval_chats = conversations[:eval_count]
train_chats = conversations[eval_count:]

MAX_LENGTH = 1024

def make_examples(chats, split_name):
    examples = []
    skipped_prefix = 0
    skipped_short = 0
    for raw_messages in chats:
        messages = normalize_chat(raw_messages)
        for index, message in enumerate(messages):
            if message["role"] != "assistant":
                continue
            history = messages[:index]
            if not any(m["role"] == "user" for m in history):
                continue

            prompt_ids = to_ids(tokenizer.apply_chat_template(
                history, tokenize=True, add_generation_prompt=True
            ))
            full_ids = to_ids(tokenizer.apply_chat_template(
                history + [message], tokenize=True, add_generation_prompt=False
            ))

            # Prompt tokenlarının tamamı tam sohbetin başlangıcı olmalı.
            if full_ids[:len(prompt_ids)] != prompt_ids:
                skipped_prefix += 1
                continue

            completion_ids = full_ids[len(prompt_ids):]
            if len(completion_ids) < 2:
                skipped_short += 1
                continue

            # Cevabı koru; gerekiyorsa eski prompt kısmını soldan kısalt.
            if len(completion_ids) >= MAX_LENGTH:
                completion_ids = completion_ids[:MAX_LENGTH - 1]
            prompt_budget = MAX_LENGTH - len(completion_ids)
            kept_prompt = prompt_ids[-prompt_budget:] if prompt_budget > 0 else []

            input_ids = kept_prompt + completion_ids
            labels = ([-100] * len(kept_prompt)) + completion_ids
            examples.append({
                "input_ids": input_ids,
                "labels": labels,
                "length": len(input_ids),
                "split": split_name,
            })

    print(split_name, "örnek:", len(examples),
          "| prefix uyuşmazlığı:", skipped_prefix,
          "| çok kısa cevap:", skipped_short)
    return examples

train_examples = make_examples(train_chats, "train")
eval_examples = make_examples(eval_chats, "eval")

assert len(train_examples) >= 100, "Yeterli eğitim örneği oluşturulamadı."
assert len(eval_examples) >= 10, "Yeterli doğrulama örneği oluşturulamadı."
assert all(any(label != -100 for label in item["labels"]) for item in train_examples[:100])

print("Eğitim örnekleri:", len(train_examples))
print("Doğrulama örnekleri:", len(eval_examples))
print("İlk örnek uzunluğu:", train_examples[0]["length"])
print("Maskelenen prompt token sayısı:", sum(x == -100 for x in train_examples[0]["labels"]))
print("Eğitilen cevap token sayısı:", sum(x != -100 for x in train_examples[0]["labels"]))

## 5. Örnekleri kontrol et

Eğitime geçmeden önce veri örneklerinin role sırasını ve cevap içeriğini kontrol et. Beklenmeyen veya bozuk örnek görürsen bu hücreden sonra eğitimi başlatma.

In [ ]:
for i, item in enumerate(train_examples[:3], 1):
    visible_ids = [t for t in item["input_ids"] if t != tokenizer.pad_token_id]
    print(f"\nÖRNEK {i} | token: {item['length']}")
    print("Son hedef metin:")
    target_ids = [t for t in item["labels"] if t != -100]
    print(tokenizer.decode(target_ids, skip_special_tokens=True)[:900])

## 6. QLoRA adaptörünü hazırla

LoRA, yalnızca aşağıdaki dil katmanlarının projeksiyonlarına eklenir. Görsel encoder eğitilmez. 4-bit temel ağırlıklar sabit kalır; öğrenilen küçük adaptör ağırlıkları güncellenir.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]
found = sorted({
    name.rsplit(".", 1)[-1]
    for name, module in model.named_modules()
    if name.rsplit(".", 1)[-1] in TARGET_MODULES
})
missing = sorted(set(TARGET_MODULES) - set(found))
print("Bulunan LoRA modülleri:", found)
assert not missing, f"Modelde bulunamayan hedef modüller: {missing}"

model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print("Eğitilebilir parametre oranı (%):", round(100 * trainable / total, 4))

## 7. Veri yükleyici ve eğitim ayarları

T4 için batch size 1, gradient accumulation 8, 1024 token ve FP16 kullanıyoruz. İlk eğitim turu 1 epoch'tur. Eğitim süresi ücretsiz Colab oturumuna göre değişebilir.

In [ ]:
import torch
from torch.utils.data import Dataset as TorchDataset
from transformers import Trainer, TrainingArguments

class AnonTorchDataset(TorchDataset):
    def __init__(self, rows):
        self.rows = rows
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, index):
        row = self.rows[index]
        return {"input_ids": row["input_ids"], "labels": row["labels"]}

class AnonCollator:
    def __init__(self, pad_id, pad_to_multiple_of=8):
        self.pad_id = int(pad_id)
        self.multiple = pad_to_multiple_of
    def __call__(self, features):
        max_len = max(len(x["input_ids"]) for x in features)
        if self.multiple:
            max_len = ((max_len + self.multiple - 1) // self.multiple) * self.multiple
        batch_size = len(features)
        input_ids = torch.full((batch_size, max_len), self.pad_id, dtype=torch.long)
        labels = torch.full((batch_size, max_len), -100, dtype=torch.long)
        attention_mask = torch.zeros((batch_size, max_len), dtype=torch.long)
        for i, item in enumerate(features):
            n = len(item["input_ids"])
            input_ids[i, :n] = torch.tensor(item["input_ids"], dtype=torch.long)
            labels[i, :n] = torch.tensor(item["labels"], dtype=torch.long)
            attention_mask[i, :n] = 1
        return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

pad_id = tokenizer.pad_token_id
assert pad_id is not None, "Tokenizer pad_token_id bulunamadı."

training_args = TrainingArguments(
    output_dir="/content/anon-ai-ministral3-checkpoints",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    max_grad_norm=0.3,
    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=0,
    save_safetensors=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=AnonTorchDataset(train_examples),
    eval_dataset=AnonTorchDataset(eval_examples),
    data_collator=AnonCollator(pad_id),
)
print("Trainer hazır.")
print("Yaklaşık optimizer adımı:", (len(train_examples) + 7) // 8)

## 8. Eğitimi başlat

Bu hücre LoRA ağırlıklarını gerçekten eğitir. Colab GPU bağlantısı kesilirse checkpoint dizininde son kayıtlar bulunur; otomatik devam ayarı sonraki sürümde ayrıca eklenebilir.

In [ ]:
torch.cuda.empty_cache()
train_result = trainer.train()
eval_result = trainer.evaluate()
print("Eğitim metrikleri:", train_result.metrics)
print("Doğrulama metrikleri:", eval_result)

## 9. ANON AI adaptörünü kaydet

ZIP dosyasına LoRA adaptörü ve tokenizer dosyaları konur. Bu dosya tek başına modelin tamamı değildir; kullanım sırasında uyumlu Ministral 3 temel modeli de gerekir.

In [ ]:
import os, shutil
ADAPTER_DIR = "/content/ANON-AI-ULTRA-Ministral3-LoRA"
trainer.model.save_pretrained(ADAPTER_DIR, safe_serialization=True)
tokenizer.save_pretrained(ADAPTER_DIR)

ZIP_PATH = shutil.make_archive(
    "/content/ANON-AI-ULTRA-Ministral3-LoRA",
    "zip",
    ADAPTER_DIR,
)
print("Adaptör ZIP:", ZIP_PATH)
print("Boyut (MB):", round(os.path.getsize(ZIP_PATH) / 1024 / 1024, 2))

## 10. Eğitilmiş ANON AI'yi test et

Beş temel testi yeniden çalıştır. Eğitim öncesi test raporuyla karşılaştır; yalnızca isim değiştirme değil, cevapların Türkçe kalitesi ve tutarlılığı da değerlendirilmelidir.

In [ ]:
import torch

def ask_anon(question, max_new_tokens=160):
    messages = [
        {"role": "system", "content": ANON_SYSTEM},
        {"role": "user", "content": question},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        return_dict=True,
        add_generation_prompt=True,
    )
    inputs = {
        key: (value.to(model.device) if torch.is_tensor(value) else value)
        for key, value in inputs.items()
    }
    model.eval()
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    prompt_len = inputs["input_ids"].shape[-1]
    return tokenizer.decode(
        output[0][prompt_len:],
        skip_special_tokens=True,
    ).strip()

TEST_QUESTIONS = [
    "Merhaba, nasılsın? Kısaca cevap ver.",
    "Türkiye'nin başkenti neresidir? Kısa açıkla.",
    "Yapay zekâ nedir? 12 yaşındaki birine anlat.",
    "Almanca 'Merhaba, nasılsın?' nasıl söylenir? Türkçe anlamını da yaz.",
    "Bana kendini tanıt. Adın ANON AI olsun.",
    "7'nin karesi kaçtır? Nasıl hesaplandığını göster.",
]
test_results = []
for i, question in enumerate(TEST_QUESTIONS, 1):
    answer = ask_anon(question)
    test_results.append({"question": question, "answer": answer})
    print(f"\nTEST {i}: {question}\nANON AI: {answer}")

## 11. Test raporunu ve adaptörü indir

İki dosyayı da indir. ZIP adaptör; JSON ise eğitim sonrası test sonuçlarıdır.

In [ ]:
import json
REPORT_PATH = "/content/ANON-AI-ULTRA-Ministral3-training-report.json"
with open(REPORT_PATH, "w", encoding="utf-8") as f:
    json.dump({
        "model_id": MODEL_ID,
        "gpu": torch.cuda.get_device_name(0),
        "train_examples": len(train_examples),
        "eval_examples": len(eval_examples),
        "train_metrics": train_result.metrics,
        "eval_metrics": eval_result,
        "results": test_results,
    }, f, ensure_ascii=False, indent=2, default=str)

from google.colab import files
files.download(ZIP_PATH)
files.download(REPORT_PATH)